# Autoencoder IDS Model — ISOT Drone Dataset
**Source:** University of Victoria ISOT Lab  
**Attack Types:** Benign, DoS, Injection, Ip Spoofing, Manipulation, MITM, Password Cracking, Replay, Unauth, Video  
**Features:** 61 network traffic features  
**Approach:** Train on Benign only, flag high reconstruction error as attack  
**Output:** ae_model.keras, ae_results.json

In [7]:
# ============================================================
# Cell 2: Load data
# WHAT: Load preprocessed dataset files for Autoencoder model
#
# WHY:  Autoencoder is unsupervised anomaly detection —
#       it trains ONLY on normal (benign) traffic.
#       It learns to reconstruct normal patterns.
#       High reconstruction error = attack detected.
#       label_classes needed to identify benign label number.
#
# HOW:  Step 1: load X_train, X_test, y_train, y_test
#       Step 2: load label_classes to find benign number
#       Step 3: filter X_train_normal = benign rows only
#       Step 4: AE trains on X_train_normal only
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time
import json

save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/ISOT/processed/"

X_train = np.load(save_path + "X_train.npy")
X_test  = np.load(save_path + "X_test.npy")
y_train = pd.read_csv(save_path + "y_train.csv").squeeze()
y_test  = pd.read_csv(save_path + "y_test.csv").squeeze()
label_classes = pd.read_csv(save_path + "label_classes.csv").squeeze().tolist()
feature_names = pd.read_csv(save_path + "feature_names.csv").squeeze().tolist()

# benign label number — check label_classes to confirm
print(f"Label classes: {label_classes}")
# UAV_Attack: benign=2, ISOT: Benign=0, UAVCAN: Normal=1

# filter benign rows only for training
# change the number based on dataset:
# UAV_Attack → 2, ISOT → 0, UAVCAN → 1
BENIGN_LABEL = 0

X_train_normal = X_train[y_train == BENIGN_LABEL]

print(f"X_train total:  {X_train.shape}")
print(f"X_train normal: {X_train_normal.shape}")
print(f"X_test:         {X_test.shape}")
print(f"Input features: {X_train.shape[1]}")
print("Data loaded!")

Label classes: ['Benign', 'DoS', 'Injection', 'Ip_Spoofing', 'MITM', 'Manipulation', 'Password_Cracking', 'Replay', 'Unauth', 'Video']
X_train total:  (205671, 61)
X_train normal: (89648, 61)
X_test:         (88145, 61)
Input features: 61
Data loaded!


In [2]:
# ============================================================
# Cell 3 Build Autoencoder:
# WHAT: Build Autoencoder architecture
#
# WHY:  Autoencoder learns to compress then reconstruct
#       normal traffic. When it sees attack traffic,
#       reconstruction error is high — flagged as anomaly.
#       Unlike RF/CNN, no attack labels needed for training.
#       This simulates real-world where attacks are unknown.
#
# HOW:  Encoder: input → 32 → 16 (compress)
#       Decoder: 16 → 32 → input (reconstruct)
#       Loss = MSE (mean squared error between input and output)
#       Low MSE = normal traffic (model reconstructs well)
#       High MSE = attack traffic (model struggles to reconstruct)
# ============================================================

print("Building Autoencoder...")

input_dim = X_train.shape[1]  # auto-detect feature count

# Encoder
input_layer = keras.Input(shape=(input_dim,))
encoded = keras.layers.Dense(32, activation='relu')(input_layer)
encoded = keras.layers.Dense(16, activation='relu')(encoded)

# Decoder
decoded = keras.layers.Dense(32, activation='relu')(encoded)
decoded = keras.layers.Dense(input_dim, activation='linear')(decoded)

# Autoencoder model
autoencoder = keras.Model(input_layer, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.summary()
print("Autoencoder built!")

Building Autoencoder...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 61)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 61)             │         2,013 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,069 (19.80 KB)

 Trainable params: 5,069 (19.80 KB)

 Non-trainable params: 0 (0.00 B)

Autoencoder built!


In [3]:
# ============================================================
# Cell 4 Train:
# WHAT: Train Autoencoder on NORMAL traffic only
#
# WHY:  Training on benign only forces the model to learn
#       what normal looks like. It cannot learn attack patterns.
#       When attack traffic appears, reconstruction fails
#       producing high MSE — our anomaly signal.
#       Training time recorded as SE metric.
#
# HOW:  input = output (X_train_normal → X_train_normal)
#       epochs=10 = 10 full passes through normal data
#       batch_size=512 = process 512 samples at a time
#       validation_split=0.1 = monitor overfitting
# ============================================================

print("Training Autoencoder on normal traffic only...")
start_time = time.time()

history = autoencoder.fit(
    X_train_normal, X_train_normal,  # input = output
    epochs=10,
    batch_size=512,
    validation_split=0.1,
    verbose=1
)

ae_train_time = round(time.time() - start_time, 2)
print(f"Training complete! Time: {ae_train_time}s")

Training Autoencoder on normal traffic only...
Epoch 1/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.3261 - val_loss: 0.0945
Epoch 2/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0809 - val_loss: 0.0475
Epoch 3/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0432 - val_loss: 0.0144
Epoch 4/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0101 - val_loss: 0.0064
Epoch 5/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0071 - val_loss: 0.0066
Epoch 6/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0060 - val_loss: 0.0043
Epoch 7/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0043 - val_loss: 0.0036
Epoch 8/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0035 - val_loss: 0.0029
Epoch 9/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0030 - val_loss: 0.0028
Epoch 10/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0037 - val_loss: 0.0024
Training complete! Time: 10.97s


In [4]:
# ============================================================
# Cell 5 Calculate reconstruction error:
# WHAT: Calculate reconstruction error (MSE) for all test samples
#
# WHY:  MSE measures how well AE reconstructs each sample.
#       Normal traffic → low MSE (AE learned this pattern)
#       Attack traffic → high MSE (AE never saw this pattern)
#       We use 95th percentile of normal MSE as threshold.
#       Above threshold = predicted attack.
#
# HOW:  Step 1: predict() → reconstruct all test samples
#       Step 2: MSE = mean((original - reconstructed)^2)
#       Step 3: get normal test MSE only
#       Step 4: threshold = 95th percentile of normal MSE
# ============================================================

print("Calculating reconstruction errors...")
start_time = time.time()

X_pred = autoencoder.predict(X_test, batch_size=512, verbose=1)
mse = np.mean(np.power(X_test - X_pred, 2), axis=1)

ae_pred_time = round(time.time() - start_time, 2)

print(f"MSE shape: {mse.shape}")
print(f"Mean MSE:  {np.mean(mse):.4f}")
print(f"Max MSE:   {np.max(mse):.4f}")
print(f"Min MSE:   {np.min(mse):.4f}")
print("Reconstruction errors calculated!")

Calculating reconstruction errors...
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
MSE shape: (88145,)
Mean MSE:  0.6631
Max MSE:   5363.8765
Min MSE:   0.0001
Reconstruction errors calculated!


In [5]:
# ============================================================
# Cell 6 Find threshold and evaluate:
# WHAT: Set anomaly threshold and evaluate model performance
#
# WHY:  95th percentile of normal MSE = threshold
#       meaning 95% of normal traffic is below threshold
#       Above threshold = predicted attack
#       This converts AE reconstruction error into binary prediction
#       We compare against true binary labels (benign vs attack)
#
# HOW:  Step 1: get MSE for normal test samples only
#       Step 2: threshold = 95th percentile
#       Step 3: y_pred_binary = 1 if MSE > threshold else 0
#       Step 4: y_true_binary = 1 if not benign else 0
#       Step 5: calculate Accuracy, Precision, Recall, F1
# ============================================================

y_test_reset = y_test.reset_index(drop=True)

# get normal MSE for threshold calculation
normal_mse = mse[y_test_reset == BENIGN_LABEL]
threshold = np.percentile(normal_mse, 95)
print(f"Threshold (95th percentile): {threshold:.4f}")

# binary predictions
y_pred_binary = (mse > threshold).astype(int)   # 1=attack, 0=normal
y_true_binary = (y_test_reset != BENIGN_LABEL).astype(int)  # 1=attack, 0=normal

# calculate metrics
acc = accuracy_score(y_true_binary, y_pred_binary)
p   = precision_score(y_true_binary, y_pred_binary)
r   = recall_score(y_true_binary, y_pred_binary)
f1  = f1_score(y_true_binary, y_pred_binary)

print(f"\n=== Autoencoder Results ===")
print(f"Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {p:.4f} ({p*100:.2f}%)")
print(f"Recall:    {r:.4f} ({r*100:.2f}%)")
print(f"F1 Score:  {f1:.4f} ({f1*100:.2f}%)")
print(f"Threshold: {threshold:.4f}")
print(f"Training time:   {ae_train_time}s")
print(f"Prediction time: {ae_pred_time}s")

Threshold (95th percentile): 0.0045

=== Autoencoder Results ===
Accuracy:  0.9781 (97.81%)
Precision: 0.9628 (96.28%)
Recall:    0.9999 (99.99%)
F1 Score:  0.9810 (98.10%)
Threshold: 0.0045
Training time:   10.97s
Prediction time: 2.18s


In [6]:
# ============================================================
# Cell 7 Save:
# WHAT: Save trained AE model and results to disk
#
# WHY:  Saving model means no need to retrain for SHAP/LIME/PI
#       JSON results used for paper tables and comparisons
#       Same save format across all datasets for consistency
#
# HOW:  Step 1: save model as .keras file
#       Step 2: save metrics as JSON file
# ============================================================

autoencoder.save(save_path + "ae_model.keras")

ae_results = {
    "model": "Autoencoder",
    "dataset": "ISOT", 
    "accuracy": round(acc, 4),
    "precision": round(p, 4),
    "recall": round(r, 4),
    "f1_score": round(f1, 4),
    "threshold": round(float(threshold), 6),
    "training_time_seconds": ae_train_time,
    "prediction_time_seconds": ae_pred_time,
    "epochs": 10,
    "batch_size": 512
}

with open(save_path + "ae_results.json", "w") as f:
    json.dump(ae_results, f, indent=4)

print("Model saved: ae_model.keras")
print("Results saved: ae_results.json")
print(f"\nResults summary:")
for k, v in ae_results.items():
    print(f"  {k}: {v}")

Model saved: ae_model.keras
Results saved: ae_results.json

Results summary:
  model: Autoencoder
  dataset: ISOT
  accuracy: 0.9781
  precision: 0.9628
  recall: 0.9999
  f1_score: 0.981
  threshold: 0.004492
  training_time_seconds: 10.97
  prediction_time_seconds: 2.18
  epochs: 10
  batch_size: 512


## Autoencoder Results Summary — ISOT Drone Dataset

| Metric | Value |
|--------|-------|
| Dataset | ISOT Drone Dataset (University of Victoria) |
| Model | Autoencoder (10 epochs, batch=512) |
| Train set | 89,648 normal rows only |
| Test set | 88,145 rows |
| Features | 61 network traffic features |
| Threshold | 0.0045 (95th percentile of normal MSE) |
| Accuracy | 97.81% |
| Precision | 96.28% |
| Recall | 99.99% |
| F1 Score | 98.10% |
| Training time | 10.97s |
| Prediction time | 2.18s |

### Key Finding
AE achieved **98.10% F1** — much better than CICIDS2017 AE (54.95%).
Near-perfect Recall (99.99%) means AE catches virtually all attacks.
Precision (96.28%) is also high — few false alarms.
ISOT network traffic features are highly discriminative even for
unsupervised anomaly detection.
Despite training on only 10%(2,938,147 rows (2.9 million), 10% sampling: 293,816 rows) of the full ISOT dataset due to disk space constraints, the Autoencoder achieved 98.10% F1, demonstrating robust anomaly detection even with limited training data.